In [ ]:
# Safe
def dynamic_safe(new_df, pool, int_var, time_name, t):
    new_df.loc[new_df[time_name] == t, int_var] = 0
    # Safe discharge: patients are very stable and ready for ICU discharge
    condition_safe = safe_discharge_mask = (
    # 1) Ventilation status
    #    Allow unknown / non-invasive / assisted modes; exclude clearly controlled invasive.
    new_df['vent_mode__last__last_12h'].isin(['unknown', 'invasive_assisted']) &

    # Vent mode unchanged for at least 2 hours (not constantly switching)
    (new_df['vent_mode__hours_since_last__last_12h'] >= 2.0) &

    # 2) Gas exchange
    # pCO2: avoid severe hypo-/hypercapnia   
    (
        new_df['pco2_arterial__mean__last_12h'].between(30.0, 60.0)
    ) &
    # pO2: avoid severe hypoxemia; high values are usually not a reason to keep in ICU
    (
        new_df['po2_arterial__mean__last_12h'].between(55.0, 250.0)
    ) &
    # O2 flow: lower support is better, but do not make it too strict
    (
        (new_df['o2_flow__last__last_12h'] <= 60.0)
    ) &
    # SpO2: reasonable saturation
    (
        new_df['o2_saturation__mean__last_12h'].between(90.0, 100.0)
    ) &

    # 3) Respiratory effort
    (
        new_df['respiratory_rate_measured__mean__last_12h'].between(8.0, 30.0)
    ) &

    # 4) Neurologic status
    # Allow some impairment but avoid deeply comatose states
    (
        new_df['glasgow_coma_scale_total__last__last_12h'] >= 10.0
    ) &

    # 5) Perfusion / metabolism
    # Lactate: keep reasonably low (your median 1.6, max 8.1)
    (
        new_df['lactate__last__last_12h'] <= 3.0
    ) &
    # Urine output: use as a weak criterion; often noisy
    (
        new_df['fluid_out_urine__mean__last_12h'] >= 30.0
    ) &

    # 6) Renal function
    # Allow substantial chronic impairment; do not require normality
    (
        new_df['ureum__last__last_12h'] <= 30.0
    ) &
    (
        new_df['creatinine__last__last_12h'] <= 300.0
    ) &

    # 7) Hemodynamics
    (
        new_df['arterial_blood_pressure_mean__mean__last_12h'].between(60.0, 110.0)
    ) &
    (
        new_df['heart_rate__mean__last_12h'].between(45.0, 130.0)
    ) &

    # 8) Hemoglobin
    (
        new_df['hemoglobin__last__last_12h'] >= 7.0
    ) &

    # 9) Temperature
    (
        new_df['temperature__mean__last_12h'].between(35.0, 38.5)
    ) &

    # 10) Coagulation (weakened)
    (
        new_df['activated_partial_thromboplastin_time__last__last_12h'].between(20.0, 80.0)
    ) &

    # 11) Acid–base
    (
        new_df['bicarbonate_arterial__last__last_12h'].between(15.0, 35.0)
    )
)
    new_df.loc[condition_safe, int_var] = 1

In [ ]:
def dynamic_safer_v2(new_df, pool, int_var, time_name, t):
    # default: do not discharge
    new_df.loc[new_df[time_name] == t, int_var] = 0

    # Safer discharge: only when clearly off invasive ventilation and physiologically stable
    condition_safer = (
        # 1) Ventilation: DO NOT discharge if status is unknown.
        # Only allow if explicitly NOT invasive.
        # (Change this allow-list to your actual non-invasive labels if present.)
        #new_df['vent_mode__last__last_12h'].isin(['unknown']) &
            # 'none', 'nasal_cannula', 'face_mask', 'high_flow', 'noninvasive',  # examples
            # If you truly only have these categories, then allow only those that are clearly non-invasive.
            # With your current stats, at minimum exclude unknown and exclude invasive modes.

        # If you don't have explicit non-invasive labels and only have the 4 shown,
        # then the safest is: allow NONE (i.e., never discharge on vent_mode alone).
        # In that case, replace the above with:
        (~new_df['vent_mode__last__last_12h'].isin(['unknown', 'invasive_assisted', 'invasive_controlled'])) &

        # Vent status not just "recently changed" (proxy for stability)
        (new_df['vent_mode__hours_since_last__last_12h'] >= 4.0) &

        # 2) Oxygenation / support: require low support and adequate saturation
        (new_df['o2_saturation__mean__last_12h'] >= 94.0) &
        (new_df['o2_flow__last__last_12h'] <= 30.0) &

        # Avoid hypoxemia (PO2 low) and severe hypercapnia
        (new_df['po2_arterial__mean__last_12h'] >= 70.0) &
        (new_df['pco2_arterial__mean__last_12h'].between(32.0, 55.0)) &

        # 3) Respiratory effort: tighten RR
        (new_df['respiratory_rate_measured__mean__last_12h'].between(10.0, 24.0)) &

        # 4) Neuro: require near-normal mentation
        (new_df['glasgow_coma_scale_total__last__last_12h'] >= 14.0) &

        # 5) Perfusion / metabolism: tighten lactate + MAP
        (new_df['lactate__last__last_12h'] <= 2.0) &
        (new_df['arterial_blood_pressure_mean__mean__last_12h'] >= 65.0) &

        # Keep tachycardia down (proxy for instability)
        (new_df['heart_rate__mean__last_12h'].between(50.0, 110.0)) &

        # 6) Renal: require reasonable urine output (use stricter threshold)
        (new_df['fluid_out_urine__mean__last_12h'] >= 50.0) &

        # Keep renal labs from extreme values (still permissive re chronic disease)
        (new_df['ureum__last__last_12h'] <= 25.0) &
        (new_df['creatinine__last__last_12h'] <= 250.0) &

        # 7) Temperature: tighter fever / hypothermia bounds
        (new_df['temperature__mean__last_12h'].between(35.5, 38.0)) &

        # 8) Acid–base: avoid substantial metabolic derangement
        (new_df['bicarbonate_arterial__last__last_12h'].between(18.0, 30.0)) &

        # 9) Coagulation: keep modestly permissive (optional gate)
        (new_df['activated_partial_thromboplastin_time__last__last_12h'].between(22.0, 70.0)) &

        # 10) Hb: keep as before (or tighten if transfusion practice makes sense)
        (new_df['hemoglobin__last__last_12h'] >= 7.5)
    )

    new_df.loc[condition_safer, int_var] = 1


In [ ]:
def dynamic_knight_stepdown(new_df, pool, int_var, time_name, t):
    new_df.loc[new_df[time_name] == t, int_var] = 0

    condition_stepdown = (
        # --- ICU-only gates (keep strict) ---

        # Ventilation: must be clearly NOT invasive.
        # Your data has many 'unknown' — treat as NOT eligible to discharge.
        (~new_df['vent_mode__last__last_12h'].isin(['unknown', 'invasive_assisted', 'invasive_controlled'])) &

        # Stability proxy: not changing vent status very recently
        (new_df['vent_mode__hours_since_last__last_12h'] >= 4.0) &

        # Oxygenation on manageable support (ward/HDU):
        # Use SpO2 + O2 flow as proxy for FiO2.
        (new_df['o2_saturation__mean__last_12h'] >= 92.0) &
        (new_df['o2_flow__last__last_12h'] <= 40.0) &

        # Avoid severe hypercapnia/acidosis physiology
        (new_df['pco2_arterial__mean__last_12h'].between(30.0, 60.0)) &
        (new_df['bicarbonate_arterial__last__last_12h'] >= 18.0) &

        # Respiratory rate: allow mild tachypnea, not severe distress
        (new_df['respiratory_rate_measured__mean__last_12h'].between(10.0, 32.0)) &

        # Hemodynamics: use MAP proxy; allow ward-manageable range
        (new_df['arterial_blood_pressure_mean__mean__last_12h'] >= 65.0) &
        (new_df['heart_rate__mean__last_12h'].between(50.0, 120.0)) &

        # Perfusion/metabolic: allow mild elevation, exclude shock
        (new_df['lactate__last__last_12h'] <= 2.5) &

        # CNS: allow mild impairment but avoid very low GCS
        (new_df['glasgow_coma_scale_total__last__last_12h'] >= 12.0) &


        # --- Ward-manageable conditions (relaxed) ---

        # Temperature: allow fever manageable on ward
        (new_df['temperature__mean__last_12h'].between(35.5, 38.5)) &

        # Hemoglobin: 7 g/dL is often manageable (transfuse/monitor outside ICU)
        (new_df['hemoglobin__last__last_12h'] >= 7.0) &

        # Renal: AKI/CKD can be managed outside ICU; keep only extreme outliers out
        (new_df['creatinine__last__last_12h'] <= 350.0) &
        (new_df['ureum__last__last_12h'] <= 35.0) &

        # Urine output: keep as weak guardrail (optional)
        (new_df['fluid_out_urine__mean__last_12h'] >= 20.0)
    )

    new_df.loc[condition_stepdown, int_var] = 1


In [ ]:
def dynamic_discharge_by_default(new_df, pool, int_var, time_name, t):
    # default: discharge
    new_df.loc[new_df[time_name] == t, int_var] = 1

    # ICU-only "red flags" (if any present => keep in ICU)
    red_flags = (
        # A) Clear invasive ventilation (block)
        new_df['vent_mode__last__last_12h'].isin(['invasive_assisted', 'invasive_controlled']) |

        # B) Severe oxygenation / support (block)
        (new_df['o2_saturation__mean__last_12h'] < 88.0) |
        (new_df['o2_flow__last__last_12h'] > 80.0) |   # very high support only

        # C) Severe respiratory failure physiology (block)
        (new_df['pco2_arterial__mean__last_12h'] > 75.0) |
        (new_df['respiratory_rate_measured__mean__last_12h'] > 40.0) |

        # D) Shock / severe hemodynamic instability (block)
        (new_df['arterial_blood_pressure_mean__mean__last_12h'] < 55.0) |
        (new_df['heart_rate__mean__last_12h'] > 155.0) |
        (new_df['lactate__last__last_12h'] > 4.0) |

        # E) Deep coma (block)
        (new_df['glasgow_coma_scale_total__last__last_12h'] < 8.0) |

        # F) Severe acid–base derangement (block)
        (new_df['bicarbonate_arterial__last__last_12h'] < 12.0) |

        # G) Extreme temperature (block)
        (new_df['temperature__mean__last_12h'] < 34.0) |
        (new_df['temperature__mean__last_12h'] > 39.5) |

        # H) Very low urine output (block; keep weak and only extreme)
        (new_df['fluid_out_urine__mean__last_12h'] < 10.0)
    )

    # Handle vent_mode == 'unknown':
    # Do NOT automatically block. Instead block only if physiology suggests high support/instability.
    unknown_vent_high_risk = (
        (new_df['vent_mode__last__last_12h'] == 'unknown') &
        (
            (new_df['o2_flow__last__last_12h'] > 60.0) |
            (new_df['o2_saturation__mean__last_12h'] < 90.0) |
            (new_df['respiratory_rate_measured__mean__last_12h'] > 35.0) |
            (new_df['pco2_arterial__mean__last_12h'] > 65.0)
        )
    )

    red_flags = red_flags | unknown_vent_high_risk

    # Time-based relaxation: after prolonged ICU time, relax further to prevent “ICU stockpiling”
    # (You can tune these cutoffs; idea is structural: don't allow indefinite ICU stay.)
    late = (new_df[time_name] == t) & (t >= 40)  # e.g., 40 bins = 20 days if 12h bins

    red_flags_late_only = (
        # only the most extreme red flags remain after long stays
        new_df['vent_mode__last__last_12h'].isin(['invasive_controlled']) |
        (new_df['o2_saturation__mean__last_12h'] < 85.0) |
        (new_df['arterial_blood_pressure_mean__mean__last_12h'] < 50.0) |
        (new_df['lactate__last__last_12h'] > 6.0) |
        (new_df['glasgow_coma_scale_total__last__last_12h'] < 6.0)
    )

    # Apply: early/mid uses full red_flags; late uses only extreme flags
    keep_in_icu = (red_flags & ~late) | (red_flags_late_only & late)

    new_df.loc[keep_in_icu, int_var] = 0
